In [1]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.layers import Input, Dense, Reshape, Flatten, Concatenate, Embedding
from tensorflow.keras.layers import Conv2D, Conv2DTranspose, LeakyReLU
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
import datetime
import io
from tensorflow.keras.callbacks import TensorBoard


2025-07-27 17:02:11.121151: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-07-27 17:02:11.126560: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-07-27 17:02:11.139973: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1753628531.160431   29694 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1753628531.165880   29694 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1753628531.179474   29694 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linkin

In [2]:
# --- 1. Load and Preprocess Data ---
(x_train, y_train), (_, _) = tf.keras.datasets.cifar10.load_data()
x_train = (x_train.astype('float32') - 127.5) / 127.5

# --- 2. Define Model Constants and Hyperparameters ---
IMG_SHAPE = (32, 32, 3)
NUM_CLASSES = 10
LATENT_DIM = 100
EPOCHS = 50000
BATCH_SIZE = 64
SAMPLE_INTERVAL = 2000
LEARNING_RATE = 0.0002
BETA_1 = 0.5



In [3]:
# --- 3. Build Generator and Discriminator Models ---
def build_generator():
    noise_input = Input(shape=(LATENT_DIM,), name="gen_noise_input")
    label_input = Input(shape=(1,), name="gen_label_input")
    label_embedding = Embedding(NUM_CLASSES, 50)(label_input)
    label_embedding = Dense(8 * 8)(label_embedding)
    label_embedding = Reshape((8, 8, 1))(label_embedding)
    noise = Dense(128 * 8 * 8, activation='relu')(noise_input)
    noise = Reshape((8, 8, 128))(noise)
    concatenated_input = Concatenate()([noise, label_embedding])
    x = Conv2DTranspose(128, kernel_size=4, strides=2, padding='same', activation='relu')(concatenated_input)
    x = Conv2DTranspose(128, kernel_size=4, strides=2, padding='same', activation='relu')(x)
    x = Conv2D(3, kernel_size=5, padding='same', activation='tanh')(x)
    generator = Model([noise_input, label_input], x, name="generator")
    return generator

def build_discriminator():
    img_input = Input(shape=IMG_SHAPE, name="disc_img_input")
    label_input = Input(shape=(1,), name="disc_label_input")
    label_embedding = Embedding(NUM_CLASSES, 50)(label_input)
    label_embedding = Dense(IMG_SHAPE[0] * IMG_SHAPE[1])(label_embedding)
    label_embedding = Reshape((IMG_SHAPE[0], IMG_SHAPE[1], 1))(label_embedding)
    concatenated_input = Concatenate()([img_input, label_embedding])
    x = Conv2D(64, kernel_size=3, strides=2, padding='same')(concatenated_input)
    x = LeakyReLU(negative_slope=0.2)(x)
    x = Conv2D(128, kernel_size=3, strides=2, padding='same')(x)
    x = LeakyReLU(negative_slope=0.2)(x)
    x = Flatten()(x)
    x = Dense(1, activation='sigmoid')(x)
    discriminator = Model([img_input, label_input], x, name="discriminator")
    return discriminator



In [4]:
# --- 4. Build and Compile Models ---
optimizer = Adam(learning_rate=LEARNING_RATE, beta_1=BETA_1)
discriminator = build_discriminator()
discriminator.compile(loss='binary_crossentropy', optimizer=optimizer, metrics=['accuracy'])

generator = build_generator()
discriminator.trainable = False

noise_input = Input(shape=(LATENT_DIM,))
label_input = Input(shape=(1,))
generated_img = generator([noise_input, label_input])
validity = discriminator([generated_img, label_input])

cgan = Model([noise_input, label_input], validity, name="cgan")
cgan.compile(loss='binary_crossentropy', optimizer=optimizer)



2025-07-27 17:02:15.832004: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [5]:
# --- 5. Implement Training Loop with Comprehensive TensorBoard Logging ---
def train_cgan(epochs, batch_size, sample_interval):
    # Set up TensorBoard File Writer
    log_dir = "logs/adversarial/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    summary_writer = tf.summary.create_file_writer(log_dir)

    # This callback will log model graphs and histograms automatically
    tensorboard_callback = TensorBoard(log_dir=log_dir, histogram_freq=1, write_graph=True)
    tensorboard_callback.set_model(cgan) # Associate the callback with our model

    # Log hyperparameters
    with summary_writer.as_default():
        hparams = {
            "latent_dim": LATENT_DIM, "epochs": epochs, "batch_size": batch_size,
            "learning_rate": LEARNING_RATE, "beta_1": BETA_1
        }
        tf.summary.text("hyperparameters", tf.convert_to_tensor(
            [f'| {key} | {value} |' for key, value in hparams.items()]
        ), step=0)


    valid = np.ones((batch_size, 1))
    fake = np.zeros((batch_size, 1))

    for epoch in range(epochs):
        # ... (Train Discriminator and Generator) ...
        idx = np.random.randint(0, x_train.shape[0], batch_size)
        real_imgs, labels = x_train[idx], y_train[idx]
        noise = np.random.normal(0, 1, (batch_size, LATENT_DIM))
        gen_imgs = generator.predict([noise, labels], verbose=0)
        d_loss_real = discriminator.train_on_batch([real_imgs, labels], valid)
        d_loss_fake = discriminator.train_on_batch([gen_imgs, labels], fake)
        d_loss = 0.5 * np.add(d_loss_real, d_loss_fake)

        noise = np.random.normal(0, 1, (batch_size, LATENT_DIM))
        sampled_labels = np.random.randint(0, NUM_CLASSES, batch_size).reshape(-1, 1)
        g_loss = cgan.train_on_batch([noise, sampled_labels], valid)

        if epoch % 100 == 0:
            print(f"{epoch} [D loss: {d_loss[0]:.4f}, acc.: {100*d_loss[1]:.2f}%] [G loss: {g_loss:.4f}]")
            # Log scalar metrics
            with summary_writer.as_default():
                tf.summary.scalar('discriminator_loss', d_loss[0], step=epoch)
                tf.summary.scalar('discriminator_accuracy', d_loss[1], step=epoch)
                tf.summary.scalar('generator_loss', g_loss, step=epoch)

        if epoch % sample_interval == 0:
            # Log generated images and weight histograms
            sample_and_log_images(epoch, generator, summary_writer)
            with summary_writer.as_default():
                for layer in discriminator.layers:
                    for weight in layer.weights:
                        tf.summary.histogram(f'disc_{layer.name}_{weight.name}', weight, step=epoch)
                for layer in generator.layers:
                    for weight in layer.weights:
                        tf.summary.histogram(f'gen_{layer.name}_{weight.name}', weight, step=epoch)
        #at the end of each epoch, log the weight histograms
        tensorboard_callback.on_epoch_end(epoch, {'d_loss': d_loss[0], 'g_loss': g_loss})               

def sample_and_log_images(epoch, generator, writer):
    r, c = 2, 5
    noise = np.random.normal(0, 1, (r * c, LATENT_DIM))
    sampled_labels = np.arange(0, 10).reshape(-1, 1)
    gen_imgs = generator.predict([noise, sampled_labels], verbose=0)
    gen_imgs = 0.5 * gen_imgs + 0.5 # Rescale to [0, 1] for viewing

    # Log images to TensorBoard
    with writer.as_default():
        tf.summary.image("Generated Images", gen_imgs, max_outputs=r*c, step=epoch)



In [10]:
# --- 6. Start Training ---
#train_cgan(epochs=EPOCHS, batch_size=BATCH_SIZE, sample_interval=SAMPLE_INTERVAL)

# --- 7. Launch TensorBoard ---
#%load_ext tensorboard
%tensorboard --logdir /home/taz/Documents/Masters/DeepLearning/Deep-Learning/DataAugmentationConditionalGan/kaggle_outputs